## Data Wrangling - Extension (Label Encoding Original Dataset to perform SHAP analysis on each feature)

We need to modify our original dataset that we fed into our selected Catboost Model to evaluate the influence of each feature towards smoking prediction. Some things we performed to guide necessary dataset transformations were -

1. Import dataset as train/test splits
2. Verify if all categorical features (ordinal and nominal) encodings made sense and preserved its meaning.
3. Evaluate if dropping values of 7 or 9 which represented NaN or Missing values reduced the data size significantly.
4. Generate a summary of rows with missing values in decreasing order of missingness count

**Outcomes/Observations**<br>

1. Large number of 7/9 values that don't given us any information.<br>
Ans. Let's encode 7 and 9 to a single value 9 and label it as "Unknown Value"

2. Some occurances of NaN needs to be handled <br>
Ans. Assign it a code of 7 like above since we don't know the value

3. Fix nominal ordering for categorical features like `GENHLTH` and `_RFHLTH` <br>
Ans. Currently the ordering for `GENHLTH` is 1> 2> 3> 4> 5> 7, we need to make it 1 < 2 < 3 < 4 < 5 < 7;
    The ordering for `_RFHLTH` is 1 > 2 > 7, we need to make it 1 < 2 < 7

4. Recode 7 to NaN, since Catboost Natively handles NaN values


In [170]:
#import dataset that was used to build catboost model, which was selected as the successful model for smoking 
#prediction at the end of Part 1 of smoker prediction

### 1. Data import from Feature Engineering Notebook
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
# Load training/ test data
#original data from data/modeling directory
# Load when needed
with open('../data/modeling/data_splits.pkl', 'rb') as f:
    data = pickle.load(f)
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']

In [171]:
#check the shape of the data
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
#(105000, 21) (45000, 21) (105000,) (45000,)

(1849209, 48) (462303, 48) (1849209,) (462303,)


In [172]:
# Get category and object columns
cat_obj_features = [col for col in X_train.columns if X_train[col].dtype.name in ['category', 'object']]

ordinal_features = []
nomial_features = []

for col in cat_obj_features:
    unique_vals = X_train[col].unique()
    print(f"{col}: {unique_vals}")

    # Check if the category is ordered
    if X_train[col].dtype.name == 'category' and X_train[col].cat.ordered:
        ordinal_features.append(col)
    else:
        nomial_features.append(col)

print("Ordinal features:", ordinal_features)
print("Nomial features:", nomial_features)

DISPCODE: [1100, 1200]
Categories (2, int64): [1100, 1200]
GENHLTH: [1, 2, 3, 4, 7, 5]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 7]
PHYSHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
MENTHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
HLTHPLN1: [1, 7, 2]
Categories (3, int64): [1, 2, 7]
PERSDOC2: [1, 2, 7, 3]
Categories (4, int64): [1, 2, 3, 7]
MEDCOST: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHECKUP1: [4, 1, 2, 3, NaN]
Categories (4, int64): [1 < 2 < 3 < 4]
CVDINFR4: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CVDCRHD4: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CVDSTRK3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
ASTHMA3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCSCNCR: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCOCNCR: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCCOPD1: [2, 7, 1]
Categories (3, int64): [1, 2, 7]
HAVARTH3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
ADDEPEV2: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCKIDNY: [2, 1, 7]
Categories (3, 

In [173]:
# 1. Create copies of the dataframes
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()
y_train_clean = y_train.copy()
y_test_clean = y_test.copy()

# 2. Drop records with 7 or 9 in any feature columns
def drop_7_9(df):
    mask = ~df.apply(lambda row: row.isin([7, 9]).any(), axis=1)
    return mask

train_mask = drop_7_9(X_train_clean)
test_mask = drop_7_9(X_test_clean)

X_train_clean = X_train_clean[train_mask]
y_train_clean = y_train_clean[train_mask]

X_test_clean = X_test_clean[test_mask]
y_test_clean = y_test_clean[test_mask]

# 3. Print remaining records
print("X_train_clean shape:", X_train_clean.shape)
print("X_test_clean shape:", X_test_clean.shape)
print("y_train_clean shape:", y_train_clean.shape)
print("y_test_clean shape:", y_test_clean.shape)

# 4. Show percentage of loss of rows
def percent_loss(original, cleaned):
    return 100 * (1 - cleaned.shape[0] / original.shape[0])

print("X_train loss (%):", percent_loss(X_train, X_train_clean))
print("X_test loss (%):", percent_loss(X_test, X_test_clean))
print("y_train loss (%):", percent_loss(y_train, y_train_clean))
print("y_test loss (%):", percent_loss(y_test, y_test_clean))

X_train_clean shape: (729828, 48)
X_test_clean shape: (183464, 48)
y_train_clean shape: (729828,)
y_test_clean shape: (183464,)
X_train loss (%): 60.53296301283413
X_test loss (%): 60.315204530362124
y_train loss (%): 60.53296301283413
y_test loss (%): 60.315204530362124


In [174]:
# For X_train
train_missing_counts = X_train.apply(lambda row: row.isin([7, 9]).sum(), axis=1)
train_missing_table = train_missing_counts.value_counts().sort_index(ascending=False).reset_index()
train_missing_table.columns = ['num_missing_7_9', 'num_records']
train_missing_table['percent_records'] = 100 * train_missing_table['num_records'] / len(X_train)
train_missing_table = train_missing_table.sort_values('num_missing_7_9', ascending=False)
print("X_train missing value distribution:")
print(train_missing_table)

# For X_test
test_missing_counts = X_test.apply(lambda row: row.isin([7, 9]).sum(), axis=1)
test_missing_table = test_missing_counts.value_counts().sort_index(ascending=False).reset_index()
test_missing_table.columns = ['num_missing_7_9', 'num_records']
test_missing_table['percent_records'] = 100 * test_missing_table['num_records'] / len(X_test)
test_missing_table = test_missing_table.sort_values('num_missing_7_9', ascending=False)
print("X_test missing value distribution:")
print(test_missing_table)

# Drop rows with any 7 or 9 in X_train and X_test, and update y_train and y_test accordingly
train_mask = ~train_missing_counts.astype(bool)
test_mask = ~test_missing_counts.astype(bool)

X_train_no_7_9 = X_train[train_mask]
y_train_no_7_9 = y_train[train_mask]

X_test_no_7_9 = X_test[test_mask]
y_test_no_7_9 = y_test[test_mask]

X_train missing value distribution:
    num_missing_7_9  num_records  percent_records
0                31            2         0.000108
1                30            1         0.000054
2                29            2         0.000108
3                28            1         0.000054
4                27            1         0.000054
5                26            5         0.000270
6                25            2         0.000108
7                24           12         0.000649
8                23           13         0.000703
9                22           12         0.000649
10               21           19         0.001027
11               20           30         0.001622
12               19           78         0.004218
13               18          120         0.006489
14               17           79         0.004272
15               16          105         0.005678
16               15          115         0.006219
17               14          258         0.013952
18            

We observe a couple of things, let's deal it case by case- <br>

1. Large number of 7/9 values that don't given us any information.<br>
Ans. Let's encode 7 and 9 to a single value 9 and label it as "Unknown Value"

2. Some occurances of NaN needs to be handled <br>
Ans. Assign it a code of 9 like above since we don't know the value

3. Fix nominal ordering for categorical features like `GENHLTH` and `_RFHLTH` <br>
Ans. Currently the ordering for `GENHLTH` is 1> 2> 3> 4> 5> 7, we need to make it 1 < 2 < 3 < 4 < 5 < 7;
    The ordering for `_RFHLTH` is 1 > 2 > 7, we need to make it 1 < 2 < 7

4. Recode 7 to NaN, since Catboost Natively handles NaN values



(1) Replace 9 with 7 across all categorical values

In [175]:
# Encode categorical features: replace 9 with 7 for all columns in ordinal_features and nomial_features
X_train_no_9 = X_train.copy()
X_test_no_9 = X_test.copy()

for col in ordinal_features + nomial_features:
    if col in X_train_no_9.columns:
        X_train_no_9[col] = X_train_no_9[col].replace(9, 7)
    if col in X_test_no_7_9.columns:
        X_test_no_9[col] = X_test_no_9[col].replace(9, 7)

/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/4001020775.py:7: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_train_no_9[col] = X_train_no_9[col].replace(9, 7)
/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/4001020775.py:9: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_test_no_9[col] = X_test_no_9[col].replace(9, 7)


Verify if the codes only display 7 for all Categorical columns

In [176]:
# Get category and object columns
cat_obj_features_no_9 = [col for col in X_train_no_9.columns if X_train_no_9[col].dtype.name in ['category', 'object']]

ordinal_features_no_9 = []
nominal_features_no_9 = []

for col in cat_obj_features:
    unique_vals = X_train_no_9[col].unique()
    print(f"{col}: {unique_vals}")

    # Check if the category is ordered
    if X_train_no_9[col].dtype.name == 'category' and X_train_no_9[col].cat.ordered:
        ordinal_features_no_9.append(col)
    else:
        nominal_features_no_9.append(col)

print("Ordinal features:", ordinal_features_no_9)
print("Nominal features:", nominal_features_no_9)

DISPCODE: [1100, 1200]
Categories (2, int64): [1100, 1200]
GENHLTH: [1, 2, 3, 4, 7, 5]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 7]
PHYSHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
MENTHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
HLTHPLN1: [1, 7, 2]
Categories (3, int64): [1, 2, 7]
PERSDOC2: [1, 2, 7, 3]
Categories (4, int64): [1, 2, 3, 7]
MEDCOST: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHECKUP1: [4, 1, 2, 3, NaN]
Categories (4, int64): [1 < 2 < 3 < 4]
CVDINFR4: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CVDCRHD4: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CVDSTRK3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
ASTHMA3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCSCNCR: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCOCNCR: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCCOPD1: [2, 7, 1]
Categories (3, int64): [1, 2, 7]
HAVARTH3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
ADDEPEV2: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCKIDNY: [2, 1, 7]
Categories (3, 

(2) Deal with NaN in `CHECKUP1` feature

In [177]:
X_train_no_9['CHECKUP1'].value_counts(dropna=False)

CHECKUP1
1      1340931
2       216259
4       141407
3       126895
NaN      23717
Name: count, dtype: int64

In [178]:
#Encode NaN in CHECKUP1 with 7
X_train_no_9['CHECKUP1'] = X_train_no_9['CHECKUP1'].cat.add_categories(7).fillna(7)
X_test_no_9['CHECKUP1'] = X_test_no_9['CHECKUP1'].cat.add_categories(7).fillna(7)

Verify if NaN is encoded as 7

In [179]:
X_train_no_9['CHECKUP1'].value_counts(dropna=False)

CHECKUP1
1    1340931
2     216259
4     141407
3     126895
7      23717
Name: count, dtype: int64

In [180]:
X_test_no_9['CHECKUP1'].value_counts(dropna=False)

CHECKUP1
1    335468
2     54033
4     35186
3     31707
7      5909
Name: count, dtype: int64

(3) Fix ordinal features [`GENHLTH` and `_RFHLTH`] , fix the order

In [181]:
#display the catergories in list ordinal_features_no_9
for col in ordinal_features_no_9:
    unique_vals = X_train_no_9[col].unique()
    print(f"{col}: {unique_vals}")
    print(f"{col}: {X_train_no_9[col].value_counts()}")


GENHLTH: [1, 2, 3, 4, 7, 5]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 7]
GENHLTH: GENHLTH
2    600008
3    566663
1    322971
4    250041
5    103107
7      6419
Name: count, dtype: int64
PHYSHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
PHYSHLTH: PHYSHLTH
1    1147061
2     403847
4     192475
3     105826
Name: count, dtype: int64
MENTHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
MENTHLTH: MENTHLTH
1    1252474
2     350489
4     140319
3     105927
Name: count, dtype: int64
CHECKUP1: [4, 1, 2, 3, 7]
Categories (5, int64): [1 < 2 < 3 < 4 < 7]
CHECKUP1: CHECKUP1
1    1340931
2     216259
4     141407
3     126895
7      23717
Name: count, dtype: int64
_RFHLTH: [1, 2, 7]
Categories (3, int64): [1 < 2 < 7]
_RFHLTH: _RFHLTH
1    1489642
2     353148
7       6419
Name: count, dtype: int64
_EDUCAG: [3, 2, 4, 1]
Categories (4, int64): [1 < 2 < 3 < 4]
_EDUCAG: _EDUCAG
4    655961
2    533813
3    503588
1    155847
Name: count, dtype: int64
_INCOMG: [5, 2, 4, 1, 3]


In [182]:
# Reverse the codes for GENHLTH: 1->5, 2->4, 3->3, 4->2, 5->1, and remove 7 from categories
genhlth_map = {1: 5, 2: 4, 3: 3, 4: 2, 5: 1}

# Apply mapping and remove 7 from categories
for df in [X_train_no_9, X_test_no_9]:
    # Only map codes 1-5, leave 7 as is for unknown
    df['GENHLTH'] = df['GENHLTH'].map(lambda x: genhlth_map.get(x, x))
    # Set new categories and order
    df['GENHLTH'] = df['GENHLTH'].astype('category')
    df['GENHLTH'] = df['GENHLTH'].cat.set_categories([1, 2, 3, 4, 5], ordered=True)

Verify `GENHLTH`, order and value_counts

In [183]:
print(X_train_no_9['GENHLTH'].value_counts(dropna=False))
X_train_no_9['GENHLTH'].unique()

GENHLTH
4      600008
3      566663
5      322971
2      250041
1      103107
NaN      6419
Name: count, dtype: int64


[5, 4, 3, 2, NaN, 1]
Categories (5, int64): [1 < 2 < 3 < 4 < 5]

Do the same for `_RFHLTH`

In [184]:
# Reverse the codes for _RFHLTH: 1->5, 2->4, 3->3, 4->2, 5->1, and remove 7 from categories
rfhlth_map = {1: 2, 2: 1}

# Apply mapping and remove 7 from categories
for df in [X_train_no_9, X_test_no_9]:
    # Only map codes 1-5, leave 7 as is for unknown
    df['_RFHLTH'] = df['_RFHLTH'].map(lambda x: rfhlth_map.get(x, x))
    # Set new categories and order
    df['_RFHLTH'] = df['_RFHLTH'].astype('category')
    df['_RFHLTH'] = df['_RFHLTH'].cat.set_categories([1, 2], ordered=True)

In [185]:
print(X_train_no_9['_RFHLTH'].value_counts(dropna=False))
X_train_no_9['_RFHLTH'].unique()

_RFHLTH
2      1489642
1       353148
NaN       6419
Name: count, dtype: int64


[2, 1, NaN]
Categories (2, int64): [1 < 2]

Create a copy of the X_train_no_9 and X_test_no_9, to represent codes with actual values to aid final analysis

In [186]:
X_train_for_reporting = X_train_no_9.copy()
X_test_for_reporting = X_test_no_9.copy()

(4) Recode 7 to NaN, (part of feature transformation for conducting SHAP analysis)

In [187]:

# Create new copies for recoding
X_train_with_NaN = X_train_no_9.copy()
X_test_with_NaN = X_test_no_9.copy()

# Recode value 7 in categorical columns to np.nan (unknown)
for col in cat_obj_features_no_9:
    if col in X_train_with_NaN.columns:
        X_train_with_NaN[col] = X_train_with_NaN[col].replace(7, pd.NA)
    if col in X_test_with_NaN.columns:
        X_test_with_NaN[col] = X_test_with_NaN[col].replace(7, pd.NA)


/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/159680260.py:8: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_train_with_NaN[col] = X_train_with_NaN[col].replace(7, pd.NA)
/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/159680260.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_test_with_NaN[col] = X_test_with_NaN[col].replace(7, pd.NA)


In [188]:
#display unique values again to verify
for col in nominal_features_no_9 :
    unique_vals = X_train_with_NaN[col].unique()
    print(f"{col}: {unique_vals}")
    print(f"{col}: {X_train_with_NaN[col].value_counts(dropna=False)}")

DISPCODE: [1100, 1200]
Categories (2, int64): [1100, 1200]
DISPCODE: DISPCODE
1100    1691022
1200     158187
Name: count, dtype: int64
HLTHPLN1: [1, NaN, 2]
Categories (2, int64): [1, 2]
HLTHPLN1: HLTHPLN1
1      1658655
2       184125
NaN       6429
Name: count, dtype: int64
PERSDOC2: [1, 2, NaN, 3]
Categories (3, int64): [1, 2, 3]
PERSDOC2: PERSDOC2
1      1416728
3       275919
2       150080
NaN       6482
Name: count, dtype: int64
MEDCOST: [2, 1, NaN]
Categories (2, int64): [1, 2]
MEDCOST: MEDCOST
2      1629908
1       214692
NaN       4609
Name: count, dtype: int64
CVDINFR4: [2, 1, NaN]
Categories (2, int64): [1, 2]
CVDINFR4: CVDINFR4
2      1730354
1       110146
NaN       8709
Name: count, dtype: int64
CVDCRHD4: [2, 1, NaN]
Categories (2, int64): [1, 2]
CVDCRHD4: CVDCRHD4
2      1721924
1       111454
NaN      15831
Name: count, dtype: int64
CVDSTRK3: [2, 1, NaN]
Categories (2, int64): [1, 2]
CVDSTRK3: CVDSTRK3
2      1768396
1        75808
NaN       5005
Name: count, dtype: 

Let's check all teh varialbes in our datasets one last time before moving onto SHAP analysis

In [189]:
#combine X_train_with_NaN and X_test_with_NaN into a single dataframe for reporting
X_combined_with_NaN = pd.concat([X_train_with_NaN, X_test_with_NaN], axis=0)

In [190]:
#combine y_train and y_test into a single dataframe for reporting
y_combined = pd.concat([y_train, y_test], axis=0)

In [191]:
#check shapes of data
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(X_train_with_NaN.shape, y_train.shape)
print(X_test_with_NaN.shape, y_test.shape)
print(X_combined_with_NaN.shape, y_combined.shape)


(1849209, 48) (1849209,)
(462303, 48) (462303,)
(1849209, 48) (1849209,)
(462303, 48) (462303,)
(2311512, 48) (2311512,)


In [192]:
X_combined_with_NaN.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2311512 entries, 1395283 to 1321228
Data columns (total 48 columns):
 #   Column    Dtype   
---  ------    -----   
 0   DISPCODE  category
 1   GENHLTH   category
 2   PHYSHLTH  category
 3   MENTHLTH  category
 4   HLTHPLN1  category
 5   PERSDOC2  category
 6   MEDCOST   category
 7   CHECKUP1  category
 8   CVDINFR4  category
 9   CVDCRHD4  category
 10  CVDSTRK3  category
 11  ASTHMA3   category
 12  CHCSCNCR  category
 13  CHCOCNCR  category
 14  CHCCOPD1  category
 15  HAVARTH3  category
 16  ADDEPEV2  category
 17  CHCKIDNY  category
 18  DIABETE3  category
 19  SEX       category
 20  MARITAL   category
 21  RENTHOM1  category
 22  VETERAN3  category
 23  CHILDREN  Int64   
 24  WEIGHT2   float64 
 25  HEIGHT3   float64 
 26  QLACTLM2  category
 27  USEEQUIP  category
 28  EXERANY2  category
 29  PNEUVAC3  category
 30  HIVTST6   category
 31  _RFHLTH   category
 32  _HCVU651  category
 33  _LTASTH1  category
 34  _CASTHM1  categor

In [193]:
X_combined_with_NaN.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
DISPCODE,2311512.0,2.0,1100.0,2114012.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GENHLTH,2303522.0,5.0,4.0,749356.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PHYSHLTH,2311512.0,4.0,1.0,1433703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MENTHLTH,2311512.0,4.0,1.0,1565685.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HLTHPLN1,2303494.0,2.0,1.0,2073247.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PERSDOC2,2303447.0,3.0,1.0,1771409.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MEDCOST,2305740.0,2.0,2.0,2037598.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CHECKUP1,2281886.0,4.0,1.0,1676399.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVDINFR4,2300653.0,2.0,2.0,2163066.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVDCRHD4,2291778.0,2.0,2.0,2152695.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [194]:
#save the cleaned datasets for EDA and modeling in next notebook
import pickle
with open('../data/modeling/part2/cleaned_data_splits_with_NaN.pkl', 'wb') as f:
    pickle.dump({
        'X_train_with_NaN': X_train_with_NaN,
        'X_test_with_NaN': X_test_with_NaN,
        'y_train_clean': y_train,
        'y_test_clean': y_test,
        'X_combined_with_NaN': X_combined_with_NaN,
        'y_combined': y_combined
    }, f)

In [195]:
#save nominal and ordinal features for future use
with open('../data/modeling/part2/nominal_ordinal_features.pkl', 'wb') as f:
    pickle.dump({
        'nominal_features': nominal_features_no_9,
        'ordinal_features': ordinal_features_no_9
    }, f)

In [196]:
#check shapes of data
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(X_train_with_NaN.shape, y_train.shape)
print(X_test_with_NaN.shape, y_test.shape)
print(X_combined_with_NaN.shape, y_combined.shape)


(1849209, 48) (1849209,)
(462303, 48) (462303,)
(1849209, 48) (1849209,)
(462303, 48) (462303,)
(2311512, 48) (2311512,)
